# トヨタ関連公式サイト スクレイピング結果の統合

`data/derived/toyota_official_scrape/`配下に個別保存した4つのスクレイピング結果
（トヨタ自動車未来創成センター／豊田中央研究所／AISIN IMRA日本語版／AISIN IMRA(imra.com)）を、
共通スキーマの論文単位テーブルに統合する。

**含まれないもの**: AISIN IMRA欧州版(imra.eu)はニュース記事形式（一覧ページ化されておらずページごとにタイトル・著者等が自由記述）で、
かつSSL証明書エラーがあり、他3件より格段に手間がかかるため今回は未実施（別途設計が必要、`log.md`に記載）。

**このノートブックでは著者単位への集約・OpenAlex ID紐付けは行わない**（次のフェーズ）。あくまで論文単位のソース統合のみ。

In [1]:
from pathlib import Path
import re

import pandas as pd

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'data').is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError('リポジトリ直下を特定できません。リポジトリ内で実行してください。')

SRC_DIR = ROOT / 'data' / 'derived' / 'toyota_official_scrape'

DOI_PATTERN = re.compile(r'doi\.org/(.+)$', re.IGNORECASE)


def extract_doi(link):
    if not isinstance(link, str):
        return None
    m = DOI_PATTERN.search(link)
    return m.group(1).lower() if m else None

## 1. 各ソースを共通スキーマに変換

共通列: `source_org`（データ元組織）・`title`・`authors_raw`・`venue`・`year`・`link`・`doi`・`toyota_affiliated_authors`（分かる場合のみ）

In [2]:
frontier = pd.read_csv(SRC_DIR / 'frontier_research_center.csv')
frontier_std = pd.DataFrame({
    'source_org': 'トヨタ自動車未来創成センター',
    'title': frontier['title'],
    'authors_raw': frontier['authors_display'],
    'venue': frontier['source'],
    'year': frontier['year'],
    'link': frontier['link'],
    'toyota_affiliated_authors': frontier['toyota_affiliated_authors'],
})

tytlabs = pd.read_csv(SRC_DIR / 'toyota_central_rd_labs.csv')
tytlabs_std = pd.DataFrame({
    'source_org': '豊田中央研究所',
    'title': tytlabs['title'],
    'authors_raw': tytlabs['authors_raw'],
    'venue': tytlabs['journal'],
    'year': tytlabs['year'],
    'link': tytlabs['link'],
    'toyota_affiliated_authors': None,  # このサイトには太字マーキングがない
})

aisin_jp = pd.read_csv(SRC_DIR / 'aisin_imra_jp.csv')
aisin_jp_std = pd.DataFrame({
    'source_org': 'AISIN IMRA(日本・aisin.com)',
    'title': aisin_jp['title'],
    'authors_raw': aisin_jp['authors_raw'],
    'venue': aisin_jp['venue'],
    'year': aisin_jp['venue'].str.extract(r'((?:19|20)\d{2})')[0].astype('Float64'),
    'link': aisin_jp['link'],
    'toyota_affiliated_authors': None,
})

imra_com = pd.read_csv(SRC_DIR / 'imra_com.csv')
imra_com_std = pd.DataFrame({
    'source_org': 'AISIN IMRA(米国・imra.com)',
    'title': imra_com['title'],
    'authors_raw': None,  # このサイトには著者記載がない
    'venue': imra_com['venue'],
    'year': imra_com['venue'].str.extract(r'\(((?:19|20)\d{2})\)')[0].astype('Float64'),
    'link': imra_com['link'],
    'toyota_affiliated_authors': None,
})

combined = pd.concat([frontier_std, tytlabs_std, aisin_jp_std, imra_com_std], ignore_index=True)
combined['doi'] = combined['link'].map(extract_doi)

print('ソース別件数:')
print(combined['source_org'].value_counts())
print()
print('合計件数:', len(combined))
print('DOIあり:', combined['doi'].notna().sum())
combined.head(3)

ソース別件数:
source_org
豊田中央研究所                     355
AISIN IMRA(米国・imra.com)     313
トヨタ自動車未来創成センター              154
AISIN IMRA(日本・aisin.com)     97
Name: count, dtype: int64

合計件数: 919
DOIあり: 643


,source_org,title,authors_raw,venue,year,link,toyota_affiliated_authors,doi
0,トヨタ自動車未来創成センター,Towards the Development of a Robot for Well-Be...,"Obata, M., Iwanaga, Y., Tsuchinaga, M., Mizuta...",The 2026 ACM/IEEE International Conference on ...,2026.0,https://dl.acm.org/doi/10.1145/3776734.3794360,"Iwanaga, Y.; Tsuchinaga, M.",NaN
1,トヨタ自動車未来創成センター,Surface vs. Structural Similarity: Aligning Co...,"Sone, Y., Ninomiya, Y., Miwa, K., Sumi, Y., Na...",21st The International Conference on Computer ...,2026.0,#,"Sumi, Y.; Nakanishi, R.; Mitsuda, E.; Sato, K....",NaN
2,トヨタ自動車未来創成センター,VDoF-MPC: Variable Degrees-of-Freedom Whole-Bo...,"Jin, T., Kobayashi, T., & Doi, M.",IEEE Access,2026.0,https://ieeexplore.ieee.org/document/11359690,"Doi, M",NaN


## 2. 重複チェック

AISIN IMRAの日本語版(aisin.com)と米国版(imra.com)は別法人・別窓口だが、同一論文が両方に載る可能性を確認する
（タイトル正規化一致・DOI一致の両方で確認）。

In [3]:
def norm_title(t):
    if not isinstance(t, str):
        return None
    return t.lower().strip().rstrip('.').replace('"', '').replace('\u201c', '').replace('\u201d', '')

combined['title_norm'] = combined['title'].map(norm_title)

dup_by_doi = combined[combined['doi'].notna()].duplicated(subset='doi', keep=False)
dup_by_title = combined[combined['title_norm'].notna()].duplicated(subset='title_norm', keep=False)

print('DOI重複行数:', dup_by_doi.sum())
print('タイトル正規化重複行数:', dup_by_title.sum())

if dup_by_title.sum() > 0:
    combined[combined['title_norm'].notna()][dup_by_title][['source_org', 'title', 'doi']]

DOI重複行数: 4
タイトル正規化重複行数: 12


## 3. 保存

In [4]:
out_path = SRC_DIR / 'combined_papers.csv'
combined.drop(columns=['title_norm']).to_csv(out_path, index=False)
print(f'保存: {out_path.relative_to(ROOT)}')
print(f'合計 {len(combined)} 件（4ソース統合、imra.euは未実施）')

保存: data/derived/toyota_official_scrape/combined_papers.csv
合計 919 件（4ソース統合、imra.euは未実施）
